In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import pickle
import json
from tqdm import tqdm
import os
import copy

import nomad.io.base as loader
import nomad.city_gen as cg
import nomad.traj_gen as tg
from nomad.traj_gen import Agent, Population
from nomad.city_gen import City

import nomad.data as data_folder
from pathlib import Path
data_dir = Path(data_folder.__file__).parent
path = data_dir / "garden_city.gpkg"

## Load city and configure destination diaries

In [2]:
# Load the city
city_file = data_dir / "garden-city.gpkg"
city = cg.City.from_geopackage(city_file)
start = '2024-06-01 00:00-04:00'

#option 1: symmetric
start_time = pd.date_range(start=start, periods=2, freq='180min')
unix_timestamp = [int(t.timestamp()) for t in start_time]
duration = [180]*2  # in minutes
location = ['w-x17-y10'] + ['r-x19-y11']

destinations = pd.DataFrame(
    {
        "datetime":start_time,
         "timestamp":unix_timestamp,
         "duration":duration,
         "location":location
    }
)
destinations.to_csv("exp_2_stops.csv", index=False)

## Config files for simulations
This could be in a json or yaml and should be passable to a Population object.

In [3]:
ha_values = [15/15, 17/15, 13/15]
beta_values = [5, 6, 7, 8]

N_reps = 10
sparsity_samples = 1
base_config = dict(
    dt=0.2,
    N=10,
    name_count=3,
    name_seed=2025,
    city_file=str(data_dir / "garden-city.gpkg"),
    buildings_file=str(data_dir / "garden-city-buildings-mercator.parquet"),
    destination_diary_file='exp_2_stops.csv',
    output_files=dict(
        sparse_path='./sparse_default',
        diaries_path='./diaries_default',
        homes_path='./homes_default'
    ),
    agent_params=dict(
        agent_homes='h-x14-y11',
        agent_workplaces='w-x17-y8',
        seed_trajectory=list(range(N_reps * sparsity_samples)),
        seed_sparsity=list(range(N_reps * sparsity_samples)),
        beta_ping=7,
        beta_durations=None,
        beta_start=None,
        ha=15/15
    )
)

for ha in ha_values:
    for beta in beta_values:
        cfg = copy.deepcopy(base_config)
        key = f"ha{round(ha*15)}_beta{beta}".replace('.', '_')
        cfg['agent_params']['ha'] = ha
        cfg['agent_params']['beta_ping'] = beta
        cfg['output_files']['sparse_path'] = f'./sparse_{key}'
        cfg['output_files']['diaries_path'] = f'./diaries_{key}'
        cfg['output_files']['homes_path'] = f'./homes_{key}'
        
        json_name = f'config_{key}.json'
        with open(json_name, 'w', encoding='utf-8') as f:
            json.dump(cfg, f, ensure_ascii=False, indent=4)
        print(json_name)

config_ha15_beta5.json
config_ha15_beta6.json
config_ha15_beta7.json
config_ha15_beta8.json
config_ha17_beta5.json
config_ha17_beta6.json
config_ha17_beta7.json
config_ha17_beta8.json
config_ha13_beta5.json
config_ha13_beta6.json
config_ha13_beta7.json
config_ha13_beta8.json


In [ ]:
N_reps = 500
sparsity_samples = 1
config = dict(
    dt = 0.2,
    N = 500,
    name_count=3,
    name_seed=2025,
    city_file=str(data_dir / "garden-city.gpkg"),
    buildings_file=str(data_dir / "garden-city-buildings-mercator.parquet"),
    destination_diary_file='exp_2_stops.csv',
    output_files = dict(
        sparse_path='./sparse_default', # <<<<<
        diaries_path='./diaries_default', 
        homes_path='./homes_default' 
    ),
    agent_params = dict(
        agent_homes='h-x14-y11',
        agent_workplaces='w-x17-y8',
        seed_trajectory=list(range(N_reps*sparsity_samples)),
        seed_sparsity= list(range(N_reps*sparsity_samples)),
        beta_ping= np.repeat(7, N_reps).tolist(), # <<<<<
        beta_durations=None,
        beta_start=None,
        ha=15/15 # <<<<<
    )
)

with open('config_2_stops.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=4)

In [ ]:
# option 2 (more sparse)
key = 'beta_5'
json_name = 'config_'+key+'.json'

config['N'] = 500
config['sparse_path'] = 'sparse_'+key
config['agent_params']['beta_ping'] = 5
config['agent_params']['ha'] = 17/15

with open(json_name, 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=4)

print(json_name)

config_beta_5.json


In [ ]:
key = 'beta_8_5'
json_name = 'config_'+key+'.json'

config['N'] = 250
config['sparse_path'] = 'sparse_'+key
config['agent_params']['beta_ping'] = 8.5
config['agent_params']['ha'] = 15/15

with open(json_name, 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=4)

print(json_name)

## Generate trajectories

In [6]:
import glob

config_files = sorted(glob.glob('config_ha*.json'))
print(f"Found {len(config_files)} configs: {config_files}")

# Load city once (same across all configs)
with open(config_files[0], 'r') as f:
    first_config = json.load(f)
city = City.from_geopackage(first_config["city_file"])
poi_data = pd.DataFrame({
    'building_id': city.buildings_gdf['id'].values,
    'x': city.buildings_gdf['door_point'].apply(lambda p: p[0]).values,
    'y': city.buildings_gdf['door_point'].apply(lambda p: p[1]).values
})

for config_file in config_files:
    print(f"\n=== Processing {config_file} ===")
    with open(config_file, 'r', encoding='utf-8') as f:
        config = json.load(f)

    destinations = pd.read_csv(config["destination_diary_file"], parse_dates=["datetime"])

    population = Population(city)
    population.generate_agents(
        N=config["N"],
        seed=config["name_seed"],
        name_count=config["name_count"],
        agent_homes=config["agent_params"]["agent_homes"],
        agent_workplaces=config["agent_params"]["agent_workplaces"]
    )

    for i, agent in enumerate(tqdm(population.roster.values(), desc="Generating trajectories")):
        agent.generate_trajectory(
            destination_diary=destinations,
            dt=config["dt"],
            seed=config["agent_params"]["seed_trajectory"][i],
            step_seed=config["agent_params"]["seed_trajectory"][i])

        agent.sample_trajectory(
            beta_ping=config["agent_params"]["beta_ping"],
            seed=config["agent_params"]["seed_sparsity"][i],
            ha=config["agent_params"]["ha"],
            pareto_prior=False,
            replace_sparse_traj=True)

    population.reproject_to_mercator(sparse_traj=True, full_traj=False, diaries=True, poi_data=poi_data)

    population.save_pop(
        sparse_path=config["output_files"]["sparse_path"],
        diaries_path=config["output_files"]["diaries_path"],
        homes_path=config["output_files"]["homes_path"],
        beta_ping=config["agent_params"]["beta_ping"],
        ha=config["agent_params"]["ha"]
    )
    print(f"Saved to {config['output_files']['sparse_path']}")

Found 12 configs: ['config_ha13_beta5.json', 'config_ha13_beta6.json', 'config_ha13_beta7.json', 'config_ha13_beta8.json', 'config_ha15_beta5.json', 'config_ha15_beta6.json', 'config_ha15_beta7.json', 'config_ha15_beta8.json', 'config_ha17_beta5.json', 'config_ha17_beta6.json', 'config_ha17_beta7.json', 'config_ha17_beta8.json']

=== Processing config_ha13_beta5.json ===


Generating trajectories:   0%|          | 0/10 [00:00<?, ?it/s]/Users/andresmondragon/nomad/nomad/city_gen.py:1575: UserWarning: shortest_paths not initialized. Auto-initializing with callable_only=True. Call compute_shortest_paths() explicitly.
  warnings.warn("shortest_paths not initialized. Auto-initializing with callable_only=True. Call compute_shortest_paths() explicitly.", UserWarning)
Generating trajectories: 100%|██████████| 10/10 [00:00<00:00, 21.02it/s]


Saved to ./sparse_ha13_beta5

=== Processing config_ha13_beta6.json ===


Generating trajectories: 100%|██████████| 10/10 [00:00<00:00, 20.41it/s]


Saved to ./sparse_ha13_beta6

=== Processing config_ha13_beta7.json ===


...

Saved to ./sparse_ha17_beta6

=== Processing config_ha17_beta7.json ===


Generating trajectories: 100%|██████████| 10/10 [00:00<00:00, 20.34it/s]


Saved to ./sparse_ha17_beta7

=== Processing config_ha17_beta8.json ===


Generating trajectories: 100%|██████████| 10/10 [00:00<00:00, 19.29it/s]

Saved to ./sparse_ha17_beta8


In [5]:
# Parameters according to the config file
with open('config_beta_5.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

# Load city and destination diary from config
city = City.from_geopackage(config["city_file"])
# Build POI data from buildings_gdf door info
poi_data = pd.DataFrame({
    'building_id': city.buildings_gdf['id'].values,
    'x': city.buildings_gdf['door_point'].apply(lambda p: p[0]).values,
    'y': city.buildings_gdf['door_point'].apply(lambda p: p[1]).values
})

destinations = pd.read_csv(config["destination_diary_file"], parse_dates=["datetime"])

population = Population(city)
population.generate_agents(
    N=config["N"],
    seed=config["name_seed"], 
    name_count=config["name_count"],
    agent_homes=config["agent_params"]["agent_homes"],
    agent_workplaces=config["agent_params"]["agent_workplaces"]
)

for i, agent in enumerate(tqdm(population.roster.values(), desc="Generating trajectories")):
    agent.generate_trajectory(
        destination_diary=destinations,
        dt=config["dt"],
        seed=config["agent_params"]["seed_trajectory"][i],
        step_seed=config["agent_params"]["seed_trajectory"][i])
    
    agent.sample_trajectory(
        beta_ping=config["agent_params"]["beta_ping"],
        seed=config["agent_params"]["seed_sparsity"][i],
        ha=config["agent_params"]["ha"],
        pareto_prior=False,
        replace_sparse_traj=True)

# Reproject all trajectories to Web Mercator at population level
print("Reprojecting trajectories to Web Mercator...")
population.reproject_to_mercator(sparse_traj=True, full_traj=False, diaries=True, poi_data=poi_data)

Generating trajectories:   0%|          | 0/1000 [00:00<?, ?it/s]/Users/andresmondragon/nomad/nomad/city_gen.py:1575: UserWarning: shortest_paths not initialized. Auto-initializing with callable_only=True. Call compute_shortest_paths() explicitly.
  warnings.warn("shortest_paths not initialized. Auto-initializing with callable_only=True. Call compute_shortest_paths() explicitly.", UserWarning)
Generating trajectories:  21%|██        | 211/1000 [00:10<00:38, 20.73it/s]


KeyboardInterrupt: 

In [7]:
# Save output files using save_pop method
print("Saving output files...")
population.save_pop(
    sparse_path=config["output_files"]["sparse_path"],
    diaries_path=config["output_files"]["diaries_path"],
    homes_path=config["output_files"]["homes_path"],
    beta_ping=config["agent_params"]["beta_ping"],
    ha=config["agent_params"]["ha"]
)
print("All output files saved successfully!")

Saving output files...
All output files saved successfully!
